> 在线讲义：[第 08 章 · 自回归生成的资源账：存一次的、每步读的、用完就扔的](https://wikieden.github.io/ai-infra-from-constraints/lessons/08-generation-budget.html) · 本书《从约束推导 AI Infra》

# 第 8 章 · 自回归生成的资源账：存一次的、每步读的、用完就扔的

配套讲义：仓库里 `ai-infra/notes/lessons/08-generation-budget.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 每 token 的 KV | $2 L n_{\mathrm{KV}} d_h b$ | `kv_bytes_per_token(36, 8, 128, 2)` | 147,456 字节 = 144 KiB；8192 个 1.125 GiB |
| 第三步 KV 追上权重 | $R_W / (c H)$ | `b_star_kv(8192)`、`b_star_kv(2048)` | 13、51 |
| 第四步 小例子 | 逐步加 | `old_reads_tokens(4, 4)`、`old_reads_bruteforce(4, 4)` | 22 |
| 第四步 例题 2-3 | 公式 | `old_reads_tokens(8192, 1024)` | 8,912,384，≈ 1224 GiB；保存 ≈ 1.27 GiB |

Qwen3-8B 的 144 KiB / token、R_W = 15,136,811,008 字节、例题 2-2 与 2-3 取自 AII 第 2 章 §2.2；H = 4、n_d = 4 是编的小例子。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
R_W = 15_136_811_008  # Qwen3-8B 一步 decode 的主要权重读取（字节）

**函数 `kv_bytes_per_token`**：每个上下文 token 的 KV：2 × L × n_kv × d_h × b（K、V 各一份）。

In [ ]:
def kv_bytes_per_token(L: int, n_kv: int, d_h: int, b: int) -> int:
    """每个上下文 token 的 KV：2 × L × n_kv × d_h × b（K、V 各一份）。"""
    return 2 * L * n_kv * d_h * b

In [ ]:
C_KV = kv_bytes_per_token(36, 8, 128, 2)  # Qwen3-8B、BF16

**函数 `b_star_kv`**：KV 读取追上权重读取的最小整数 batch：ceil(R_W / (c_KV × H))。

In [ ]:
def b_star_kv(H: int, r_w: int = R_W, c_kv: int = C_KV) -> int:
    """KV 读取追上权重读取的最小整数 batch：ceil(R_W / (c_KV × H))。"""
    return math.ceil(r_w / (c_kv * H))

**函数 `old_reads_tokens`**：生成期间累计读取的旧 KV（token·次）：n_d·H + n_d(n_d-1)/2。

In [ ]:
def old_reads_tokens(H: int, n_d: int) -> int:
    """生成期间累计读取的旧 KV（token·次）：n_d·H + n_d(n_d-1)/2。"""
    return n_d * H + n_d * (n_d - 1) // 2

**函数 `old_reads_bruteforce`**：一步一步加：第 j 次 decode 读 H + j 份。

In [ ]:
def old_reads_bruteforce(H: int, n_d: int) -> int:
    """一步一步加：第 j 次 decode 读 H + j 份。"""
    return sum(H + j for j in range(n_d))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Qwen3-8B 每个 token 的 KV")
print(f"  2 × 36 × 8 × 128 × 2 = {C_KV:,} 字节 = {C_KV / 1024:.0f} KiB")
print(f"  8192 个上下文 token = {C_KV * 8192 / 2**30:.3f} GiB")

In [ ]:
print("\n第三步：batch 多大时读 KV 多于读权重")
for H in (8192, 2048):
    print(f"  H = {H}：R_W / (c × H) = {R_W / (C_KV * H):.2f} → 最小 batch {b_star_kv(H)}")

In [ ]:
print("\n第四步：H = 4、生成 4 步（编的）")
print(f"  公式 {old_reads_tokens(4, 4)}，逐步加 {old_reads_bruteforce(4, 4)}；最终保存 {4 + 4} 份")

In [ ]:
print("\n第四步：例题 2-3（S = 0、P = 8192、G = 1025）")
n_d = 1024
reads = old_reads_tokens(8192, n_d)
print(f"  累计读取 {reads:,} token·次 = {reads * C_KV / 2**30:.0f} GiB")
print(f"  生成期间新增 {n_d * C_KV / 2**20:.0f} MiB；结束时保存 {(8192 + n_d) * C_KV / 2**30:.2f} GiB")

## 3. 代码里才看得到的

1. `b_star_kv` 用 `math.ceil`：12.53 个请求不存在，最小整数是 13。
2. 1224 GiB 和 1.27 GiB 都是二进制单位（除以 $2^{30}$）；用十进制 GB 会是 1314 GB 和 1.36 GB，原文用 GiB。
3. 结束时保存的是 $H + n_d$ 份，不是 $H + G$：最后一个输出 token 从没被送进模型。

## 4. 自己改一改

- 把 `old_reads_tokens(8192, n_d)` 的 $n_d$ 从 1024 改成 2048，看累计读取涨了几倍、保存涨了几倍。
- 把 `kv_bytes_per_token` 的 `b` 从 2 改成 1，看例题 2-3 的三个量哪些减半。
- 用 `b_star_kv` 扫 $H$ = 512、1024、4096、16384，画出 $B_*$ 随上下文长度的变化。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 问完就走不留 Q，白板常驻 K 和 V；
> 菜谱共用、订单跟人、配菜即弃；
> 车只一辆书包各背，上下文长早翻车；
> 存的一格一格长，读的一行一行涨。